# Gathering ESPN Team Statistics

## Data Collection

Scraping team statistics from ESPN including:
- Game Results
- Scoring Stats
- Assisting Stats
- Dicipline Stats
- Team Performance

### Sources
- https://www.espn.com/
- https://www.selenium.dev/documentation/webdriver/
- https://selenium-python.readthedocs.io/locating-elements.html

## Fixtures Scraper

In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time
from io import StringIO

def setup_driver():
    """
    Sets up and returns a Chrome WebDriver with optimized options.
    
    Returns:
        webdriver.Chrome: Configured Chrome WebDriver instance
    """
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Run without opening browser window
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def scrape_team_fixtures(driver, url, team_name):
    """
    Scrapes all fixture tables for a specific team from ESPN.
    Combines all monthly tables into one DataFrame.
    
    Args:
        driver: Selenium WebDriver instance
        url (str): ESPN team results URL
        team_name (str): Name of the team
    
    Returns:
        pd.DataFrame: All fixtures with added Team column, or None if failed
    """
    print(f"\nScraping {team_name}...")
    
    try:
        # Navigate to the page
        driver.get(url)
        
        # Wait for page to load
        time.sleep(5)
        
        # Get the page source and extract all tables
        page_source = driver.page_source
        tables = pd.read_html(StringIO(page_source))
        
        print(f"  Found {len(tables)} monthly tables")
        
        # Combine all monthly tables into one
        # Each table represents a different month
        all_fixtures = []
        for table in tables:
            all_fixtures.append(table)
        
        # Concatenate all monthly tables
        combined_fixtures = pd.concat(all_fixtures, ignore_index=True)
        
        # Add a column to identify which team this data belongs to
        combined_fixtures.insert(0, 'Team', team_name)
        
        print(f"  ✓ Scraped {len(combined_fixtures)} matches for {team_name}")
        
        return combined_fixtures
    
    except Exception as e:
        print(f"  ✗ Error scraping {team_name}: {e}")
        return None


def scrape_big_six_fixtures():
    """
    Main function to scrape fixtures for the Big 6 Premier League teams
    (excluding Tottenham) from ESPN for the 2025-2026 season.
    
    Returns:
        pd.DataFrame: Combined fixtures data for all teams
    """
    # Define teams with their ESPN URLs
    # Format: https://www.espn.com/soccer/team/results/_/id/{TEAM_ID}/league/ENG.1
    teams = {
        'Arsenal': 'https://www.espn.com/soccer/team/results/_/id/359/league/ENG.1',
        'Liverpool': 'https://www.espn.com/soccer/team/results/_/id/364/league/ENG.1',
        'Manchester United': 'https://www.espn.com/soccer/team/results/_/id/360/league/ENG.1',
        'Manchester City': 'https://www.espn.com/soccer/team/results/_/id/382/league/ENG.1',
        'Chelsea': 'https://www.espn.com/soccer/team/results/_/id/363/league/ENG.1'
    }
    
    # Setup driver
    driver = setup_driver()
    
    # Store all team fixtures
    all_fixtures = []
    
    try:
        print("="*60)
        print("SCRAPING FIXTURES FOR BIG 6 TEAMS")
        print("="*60)
        
        # Scrape each team
        for team_name, url in teams.items():
            fixtures_df = scrape_team_fixtures(driver, url, team_name)
            
            if fixtures_df is not None:
                all_fixtures.append(fixtures_df)
            
            # Small delay between teams to be respectful to ESPN servers
            time.sleep(2)
        
    finally:
        # Always close the driver
        driver.quit()
        print("\n✓ Browser closed")
    
    # Combine all team fixtures into one DataFrame
    if all_fixtures:
        combined_fixtures = pd.concat(all_fixtures, ignore_index=True)
        return combined_fixtures
    else:
        return None


# Execute the scraper
if __name__ == "__main__":
    print("="*60)
    print("ESPN Fixtures Scraper")
    print("Big 6 (excluding Tottenham) - 2025/2026 Season")
    print("="*60)
    print()
    
    # Run the scraper
    fixtures_df = scrape_big_six_fixtures()
    
    # Display and save results
    if fixtures_df is not None:
        print("\n" + "="*60)
        print("SCRAPING COMPLETE")
        print("="*60)
        print(f"\nTotal matches scraped: {len(fixtures_df)}")
        print(f"Teams: {fixtures_df['Team'].nunique()}")
        print(f"\nMatches per team:")
        print(fixtures_df['Team'].value_counts().to_string())
        
        # Show sample data
        print("\nSample data (first 5 rows):")
        print(fixtures_df.head().to_string())
        
        # Save to CSV
        output_file = 'big_six_fixtures_espn_2025_26.csv'
        fixtures_df.to_csv(output_file, index=False)
        print(f"\n✓ Data saved to: {output_file}")
        
        # Show column names
        print(f"\nColumns in dataset:")
        for col in fixtures_df.columns:
            print(f"  - {col}")
    else:
        print("\n✗ Failed to scrape fixtures data.")

ESPN Fixtures Scraper
Big 6 (excluding Tottenham) - 2025/2026 Season

SCRAPING FIXTURES FOR BIG 6 TEAMS

Scraping Arsenal...
  Found 4 monthly tables
  ✓ Scraped 12 matches for Arsenal

Scraping Liverpool...
  Found 4 monthly tables
  ✓ Scraped 12 matches for Liverpool

Scraping Manchester United...
  Found 4 monthly tables
  ✓ Scraped 11 matches for Manchester United

Scraping Manchester City...
  Found 4 monthly tables
  ✓ Scraped 12 matches for Manchester City

Scraping Chelsea...
  Found 4 monthly tables
  ✓ Scraped 12 matches for Chelsea

✓ Browser closed

SCRAPING COMPLETE

Total matches scraped: 59
Teams: 5

Matches per team:
Team
Arsenal              12
Liverpool            12
Manchester City      12
Chelsea              12
Manchester United    11

Sample data (first 5 rows):
      Team         DATE MATCH MATCH.1 MATCH.2 RESULT             COMPETITION
0  Arsenal  Sun, Nov 23   ARS   4 - 1     TOT     FT  English Premier League
1  Arsenal   Sat, Nov 8   SUN   2 - 2     ARS     F

In [4]:
Arsenal_fixtures_df

,Team,DATE,MATCH,MATCH.1,MATCH.2,RESULT,COMPETITION
0,Arsenal,"Sun, Nov 23",ARS,4 - 1,TOT,FT,English Premier League
1,Arsenal,"Sat, Nov 8",SUN,2 - 2,ARS,FT,English Premier League
2,Arsenal,"Sat, Nov 1",BUR,0 - 2,ARS,FT,English Premier League
3,Arsenal,"Sun, Oct 26",ARS,1 - 0,CRY,FT,English Premier League
4,Arsenal,"Sat, Oct 18",FUL,0 - 1,ARS,FT,English Premier League
5,Arsenal,"Sat, Oct 4",ARS,2 - 0,WHU,FT,English Premier League
6,Arsenal,"Sun, Sep 28",NEW,1 - 2,ARS,FT,English Premier League
7,Arsenal,"Sun, Sep 21",ARS,1 - 1,MNC,FT,English Premier League
8,Arsenal,"Sat, Sep 13",ARS,3 - 0,NFO,FT,English Premier League
9,Arsenal,"Sun, Aug 31",LIV,1 - 0,ARS,FT,English Premier League


In [5]:
Liverpool_fixtures_df

,Team,DATE,MATCH,MATCH.1,MATCH.2,RESULT,COMPETITION
12,Liverpool,"Sat, Nov 22",LIV,0 - 3,NFO,FT,English Premier League
13,Liverpool,"Sun, Nov 9",MNC,3 - 0,LIV,FT,English Premier League
14,Liverpool,"Sat, Nov 1",LIV,2 - 0,AVL,FT,English Premier League
15,Liverpool,"Sat, Oct 25",BRE,3 - 2,LIV,FT,English Premier League
16,Liverpool,"Sun, Oct 19",LIV,1 - 2,MAN,FT,English Premier League
17,Liverpool,"Sat, Oct 4",CHE,2 - 1,LIV,FT,English Premier League
18,Liverpool,"Sat, Sep 27",CRY,2 - 1,LIV,FT,English Premier League
19,Liverpool,"Sat, Sep 20",LIV,2 - 1,EVE,FT,English Premier League
20,Liverpool,"Sun, Sep 14",BUR,0 - 1,LIV,FT,English Premier League
21,Liverpool,"Sun, Aug 31",LIV,1 - 0,ARS,FT,English Premier League


In [6]:
ManUnited_fixtures_df

,Team,DATE,MATCH,MATCH.1,MATCH.2,RESULT,COMPETITION
24,Manchester United,"Sat, Nov 8",TOT,2 - 2,MAN,FT,English Premier League
25,Manchester United,"Sat, Nov 1",NFO,2 - 2,MAN,FT,English Premier League
26,Manchester United,"Sat, Oct 25",MAN,4 - 2,BHA,FT,English Premier League
27,Manchester United,"Sun, Oct 19",LIV,1 - 2,MAN,FT,English Premier League
28,Manchester United,"Sat, Oct 4",MAN,2 - 0,SUN,FT,English Premier League
29,Manchester United,"Sat, Sep 27",BRE,3 - 1,MAN,FT,English Premier League
30,Manchester United,"Sat, Sep 20",MAN,2 - 1,CHE,FT,English Premier League
31,Manchester United,"Sun, Sep 14",MNC,3 - 0,MAN,FT,English Premier League
32,Manchester United,"Sat, Aug 30",MAN,3 - 2,BUR,FT,English Premier League
33,Manchester United,"Sun, Aug 24",FUL,1 - 1,MAN,FT,English Premier League


In [7]:
ManCity_fixtures_df

,Team,DATE,MATCH,MATCH.1,MATCH.2,RESULT,COMPETITION
35,Manchester City,"Sat, Nov 22",NEW,2 - 1,MNC,FT,English Premier League
36,Manchester City,"Sun, Nov 9",MNC,3 - 0,LIV,FT,English Premier League
37,Manchester City,"Sun, Nov 2",MNC,3 - 1,BOU,FT,English Premier League
38,Manchester City,"Sun, Oct 26",AVL,1 - 0,MNC,FT,English Premier League
39,Manchester City,"Sat, Oct 18",MNC,2 - 0,EVE,FT,English Premier League
40,Manchester City,"Sun, Oct 5",BRE,0 - 1,MNC,FT,English Premier League
41,Manchester City,"Sat, Sep 27",MNC,5 - 1,BUR,FT,English Premier League
42,Manchester City,"Sun, Sep 21",ARS,1 - 1,MNC,FT,English Premier League
43,Manchester City,"Sun, Sep 14",MNC,3 - 0,MAN,FT,English Premier League
44,Manchester City,"Sun, Aug 31",BHA,2 - 1,MNC,FT,English Premier League


In [8]:
Chelsea_fixtures_df

,Team,DATE,MATCH,MATCH.1,MATCH.2,RESULT,COMPETITION
47,Chelsea,"Sat, Nov 22",BUR,0 - 2,CHE,FT,English Premier League
48,Chelsea,"Sat, Nov 8",CHE,3 - 0,WOL,FT,English Premier League
49,Chelsea,"Sat, Nov 1",TOT,0 - 1,CHE,FT,English Premier League
50,Chelsea,"Sat, Oct 25",CHE,1 - 2,SUN,FT,English Premier League
51,Chelsea,"Sat, Oct 18",NFO,0 - 3,CHE,FT,English Premier League
52,Chelsea,"Sat, Oct 4",CHE,2 - 1,LIV,FT,English Premier League
53,Chelsea,"Sat, Sep 27",CHE,1 - 3,BHA,FT,English Premier League
54,Chelsea,"Sat, Sep 20",MAN,2 - 1,CHE,FT,English Premier League
55,Chelsea,"Sat, Sep 13",BRE,2 - 2,CHE,FT,English Premier League
56,Chelsea,"Sat, Aug 30",CHE,2 - 0,FUL,FT,English Premier League


## Stats Scraper

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time
from io import StringIO

def setup_driver():
    """
    Sets up and returns a Chrome WebDriver with optimized options.
    
    Returns:
        webdriver.Chrome: Configured Chrome WebDriver instance
    """
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Run without opening browser window
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def scrape_scoring_stats(driver, team_id, team_name):
    """
    Scrapes scoring statistics (top scorers and top assists) for a team.
    
    Args:
        driver: Selenium WebDriver instance
        team_id (int): ESPN team ID
        team_name (str): Name of the team
    
    Returns:
        tuple: (scorers_df, assists_df) or (None, None) if failed
    """
    url = f"https://www.espn.com/soccer/team/stats/_/id/{team_id}/league/ENG.1/view/scoring"
    
    try:
        driver.get(url)
        time.sleep(4)
        
        # Get all tables
        page_source = driver.page_source
        tables = pd.read_html(StringIO(page_source))
        
        if len(tables) < 2:
            raise Exception(f"Expected 2 tables, found {len(tables)}")
        
        # Table 1: Top Scorers (RK, Name, P, G)
        scorers_df = tables[0].copy()
        scorers_df.insert(0, 'Team', team_name)
        
        # Clean NaN values in RK column (tied ranks) - forward fill
        scorers_df['RK'] = scorers_df['RK'].ffill()
        
        # Drop any rows that are completely empty
        scorers_df = scorers_df.dropna(subset=['Name'])
        
        # Table 2: Top Assists (RK, Name, P, A)
        assists_df = tables[1].copy()
        assists_df.insert(0, 'Team', team_name)
        
        # Clean NaN values in RK column (tied ranks) - forward fill
        assists_df['RK'] = assists_df['RK'].ffill()
        
        # Drop any rows that are completely empty
        assists_df = assists_df.dropna(subset=['Name'])
        
        return scorers_df, assists_df
    
    except Exception as e:
        print(f"    ✗ Error scraping scoring stats: {e}")
        return None, None


def scrape_discipline_stats(driver, team_id, team_name):
    """
    Scrapes discipline statistics (cards) for a team.
    
    Args:
        driver: Selenium WebDriver instance
        team_id (int): ESPN team ID
        team_name (str): Name of the team
    
    Returns:
        pd.DataFrame: Discipline stats or None if failed
    """
    url = f"https://www.espn.com/soccer/team/stats/_/id/{team_id}/league/ENG.1/view/discipline"
    
    try:
        driver.get(url)
        time.sleep(4)
        
        # Get all tables
        page_source = driver.page_source
        tables = pd.read_html(StringIO(page_source))
        
        if len(tables) < 1:
            raise Exception(f"Expected 1 table, found {len(tables)}")
        
        # Table 1: Cards (RK, Name, P, yc, rc, Pts)
        discipline_df = tables[0].copy()
        discipline_df.insert(0, 'Team', team_name)
        
        # Clean NaN values in RK column (tied ranks) - forward fill
        discipline_df['RK'] = discipline_df['RK'].ffill()
        
        # Drop any rows that are completely empty
        discipline_df = discipline_df.dropna(subset=['Name'])
        
        # Clean NaN values in RK column (tied ranks) - forward fill
        discipline_df['RK'] = discipline_df['RK'].ffill()
        
        # Drop any rows that are completely empty
        discipline_df = discipline_df.dropna(subset=['Name'])
        
        return discipline_df
    
    except Exception as e:
        print(f"    ✗ Error scraping discipline stats: {e}")
        return None


def scrape_performance_stats(driver, team_id, team_name):
    """
    Scrapes performance statistics (goals records, streaks, attendance) for a team.
    
    Args:
        driver: Selenium WebDriver instance
        team_id (int): ESPN team ID
        team_name (str): Name of the team
    
    Returns:
        tuple: (goals_df, streaks_df, attendance_df) or (None, None, None) if failed
    """
    url = f"https://www.espn.com/soccer/team/stats/_/id/{team_id}/league/ENG.1/view/performance"
    
    try:
        driver.get(url)
        time.sleep(4)
        
        # Get all tables
        page_source = driver.page_source
        tables = pd.read_html(StringIO(page_source))
        
        if len(tables) < 3:
            raise Exception(f"Expected 3 tables, found {len(tables)}")
        
        # Table 1: Goals records
        goals_df = tables[0].copy()
        goals_df.insert(0, 'Team', team_name)
        
        # Drop any rows with all NaN values
        goals_df = goals_df.dropna(how='all')
        
        # Table 2: Winning/unbeaten streaks
        streaks_df = tables[1].copy()
        streaks_df.insert(0, 'Team', team_name)
        
        # Drop any rows with all NaN values
        streaks_df = streaks_df.dropna(how='all')
        
        # Table 3: Attendance
        attendance_df = tables[2].copy()
        attendance_df.insert(0, 'Team', team_name)
        
        # Drop any rows with all NaN values
        attendance_df = attendance_df.dropna(how='all')
        
        return goals_df, streaks_df, attendance_df
    
    except Exception as e:
        print(f"    ✗ Error scraping performance stats: {e}")
        return None, None, None


def scrape_team_all_stats(driver, team_id, team_name):
    """
    Scrapes all statistics (scoring, discipline, performance) for a single team.
    
    Args:
        driver: Selenium WebDriver instance
        team_id (int): ESPN team ID
        team_name (str): Name of the team
    
    Returns:
        dict: Dictionary containing all scraped DataFrames
    """
    print(f"\nScraping {team_name}...")
    
    stats = {}
    
    # Scrape scoring stats
    print("  → Scoring stats...", end=" ")
    scorers_df, assists_df = scrape_scoring_stats(driver, team_id, team_name)
    if scorers_df is not None:
        stats['scorers'] = scorers_df
        stats['assists'] = assists_df
        print(f"✓ ({len(scorers_df)} scorers, {len(assists_df)} assist leaders)")
    
    # Scrape discipline stats
    print("  → Discipline stats...", end=" ")
    discipline_df = scrape_discipline_stats(driver, team_id, team_name)
    if discipline_df is not None:
        stats['discipline'] = discipline_df
        print(f"✓ ({len(discipline_df)} players)")
    
    # Scrape performance stats
    print("  → Performance stats...", end=" ")
    goals_df, streaks_df, attendance_df = scrape_performance_stats(driver, team_id, team_name)
    if goals_df is not None:
        stats['goals_records'] = goals_df
        stats['streaks'] = streaks_df
        stats['attendance'] = attendance_df
        print("✓")
    
    print(f"  ✓ Completed {team_name}")
    
    return stats


def scrape_big_six_all_stats():
    """
    Main function to scrape all statistics for the Big 6 Premier League teams
    (excluding Tottenham) from ESPN for the 2025-2026 season.
    
    Returns:
        dict: Dictionary of DataFrames organized by stat type
    """
    # Define teams with their ESPN IDs
    teams = {
        'Arsenal': 359,
        'Liverpool': 364,
        'Manchester United': 360,
        'Manchester City': 382,
        'Chelsea': 363
    }
    
    # Setup driver
    driver = setup_driver()
    
    # Store all stats organized by type
    all_stats = {
        'scorers': [],
        'assists': [],
        'discipline': [],
        'goals_records': [],
        'streaks': [],
        'attendance': []
    }
    
    try:
        print("="*60)
        print("SCRAPING ALL STATS FOR BIG 6 TEAMS")
        print("="*60)
        
        # Scrape each team
        for team_name, team_id in teams.items():
            team_stats = scrape_team_all_stats(driver, team_id, team_name)
            
            # Add each stat type to the corresponding list
            for stat_type, df in team_stats.items():
                all_stats[stat_type].append(df)
            
            # Small delay between teams
            time.sleep(2)
        
    finally:
        # Always close the driver
        driver.quit()
        print("\n✓ Browser closed")
    
    # Combine all stats by type
    combined_stats = {}
    for stat_type, dfs in all_stats.items():
        if dfs:
            # Combine and reset index for clean DataFrames
            combined_stats[stat_type] = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
    
    return combined_stats


# Execute the scraper
if __name__ == "__main__":
    print("="*60)
    print("ESPN Complete Stats Scraper")
    print("Big 6 (excluding Tottenham) - 2025/2026 Season")
    print("="*60)
    print()
    
    # Run the scraper
    all_stats = scrape_big_six_all_stats()
    
    # Make DataFrames globally accessible
    if all_stats:
        scorers_df = all_stats['scorers']
        assists_df = all_stats['assists']
        discipline_df = all_stats['discipline']
        goals_records_df = all_stats['goals_records']
        streaks_df = all_stats['streaks']
        attendance_df = all_stats['attendance']
        
        # Display summary
        print("\n" + "="*60)
        print("SCRAPING COMPLETE")
        print("="*60)
        
        print("\nCombined DataFrames Created:")
        print("="*60)
        print(f"✓ scorers_df: {len(scorers_df)} rows")
        print(f"✓ assists_df: {len(assists_df)} rows")
        print(f"✓ discipline_df: {len(discipline_df)} rows")
        print(f"✓ goals_records_df: {len(goals_records_df)} rows")
        print(f"✓ streaks_df: {len(streaks_df)} rows")
        print(f"✓ attendance_df: {len(attendance_df)} rows")
        
        # Split all stats by team
        print("\n" + "="*60)
        print("📊 Creating Team-Specific DataFrames:")
        print("="*60)
        
        teams = ['Arsenal', 'Liverpool', 'Manchester United', 'Manchester City', 'Chelsea']
        team_prefixes = ['Arsenal', 'Liverpool', 'ManUnited', 'ManCity', 'Chelsea']
        
        for team, prefix in zip(teams, team_prefixes):
            # Create team-specific DataFrames for each stat type
            # Reset index after filtering to get clean indices
            globals()[f'{prefix}_scorers_df'] = scorers_df[scorers_df['Team'] == team].copy().reset_index(drop=True)
            globals()[f'{prefix}_assists_df'] = assists_df[assists_df['Team'] == team].copy().reset_index(drop=True)
            globals()[f'{prefix}_discipline_df'] = discipline_df[discipline_df['Team'] == team].copy().reset_index(drop=True)
            globals()[f'{prefix}_goals_records_df'] = goals_records_df[goals_records_df['Team'] == team].copy().reset_index(drop=True)
            globals()[f'{prefix}_streaks_df'] = streaks_df[streaks_df['Team'] == team].copy().reset_index(drop=True)
            globals()[f'{prefix}_attendance_df'] = attendance_df[attendance_df['Team'] == team].copy().reset_index(drop=True)
            
            print(f"\n{team}:")
            print(f"  ✓ {prefix}_scorers_df: {len(globals()[f'{prefix}_scorers_df'])} players")
            print(f"  ✓ {prefix}_assists_df: {len(globals()[f'{prefix}_assists_df'])} players")
            print(f"  ✓ {prefix}_discipline_df: {len(globals()[f'{prefix}_discipline_df'])} players")
            print(f"  ✓ {prefix}_goals_records_df: {len(globals()[f'{prefix}_goals_records_df'])} records")
            print(f"  ✓ {prefix}_streaks_df: {len(globals()[f'{prefix}_streaks_df'])} records")
            print(f"  ✓ {prefix}_attendance_df: {len(globals()[f'{prefix}_attendance_df'])} records")
        
        print("\n" + "="*60)
        print("✓ All DataFrames ready to use!")
        print("="*60)
        print("\nCOMBINED DataFrames (all teams):")
        print("  - scorers_df")
        print("  - assists_df")
        print("  - discipline_df")
        print("  - goals_records_df")
        print("  - streaks_df")
        print("  - attendance_df")
        
        print("\nTEAM-SPECIFIC DataFrames:")
        print("  Arsenal: Arsenal_scorers_df, Arsenal_assists_df, Arsenal_discipline_df, etc.")
        print("  Liverpool: Liverpool_scorers_df, Liverpool_assists_df, Liverpool_discipline_df, etc.")
        print("  Man United: ManUnited_scorers_df, ManUnited_assists_df, ManUnited_discipline_df, etc.")
        print("  Man City: ManCity_scorers_df, ManCity_assists_df, ManCity_discipline_df, etc.")
        print("  Chelsea: Chelsea_scorers_df, Chelsea_assists_df, Chelsea_discipline_df, etc.")
        
        print("\nExample - Arsenal top scorers:")
        print(globals()['Arsenal_scorers_df'].head().to_string())
    
    else:
        print("\n✗ Failed to scrape statistics data.")

ESPN Complete Stats Scraper
Big 6 (excluding Tottenham) - 2025/2026 Season

SCRAPING ALL STATS FOR BIG 6 TEAMS

Scraping Arsenal...
  → Scoring stats... ✓ (25 scorers, 25 assist leaders)
  → Discipline stats... ✓ (22 players)
  → Performance stats... ✓
  ✓ Completed Arsenal

Scraping Liverpool...
  → Scoring stats... ✓ (24 scorers, 24 assist leaders)
  → Discipline stats... ✓ (19 players)
  → Performance stats... ✓
  ✓ Completed Liverpool

Scraping Manchester United...
  → Scoring stats... ✓ (25 scorers, 25 assist leaders)
  → Discipline stats... ✓ (24 players)
  → Performance stats... ✓
  ✓ Completed Manchester United

Scraping Manchester City...
  → Scoring stats... ✓ (25 scorers, 25 assist leaders)
  → Discipline stats... ✓ (22 players)
  → Performance stats... ✓
  ✓ Completed Manchester City

Scraping Chelsea...
  → Scoring stats... ✓ (25 scorers, 25 assist leaders)
  → Discipline stats... ✓ (20 players)
  → Performance stats... ✓
  ✓ Completed Chelsea

✓ Browser closed

SCRAPING C

#### Stats Dataframe Example Using Arsenal (To Access other teams type 'Liverpool', 'ManUnited', 'ManCity', and 'Chelsea' Insead of 'Arsenal')

In [21]:
Arsenal_scorers_df

,Team,RK,Name,P,G
0,Arsenal,1.0,Viktor Gyökeres,10,4
1,Arsenal,2.0,Bukayo Saka,9,3
2,Arsenal,3.0,Martín Zubimendi,11,2
3,Arsenal,3.0,Declan Rice,11,2
4,Arsenal,3.0,Jurriën Timber,11,2
5,Arsenal,3.0,Leandro Trossard,9,2
6,Arsenal,7.0,Gabriel Magalhães,11,1
7,Arsenal,7.0,Riccardo Calafiori,11,1
8,Arsenal,7.0,Mikel Merino,10,1
9,Arsenal,7.0,Eberechi Eze,10,1


In [22]:
Arsenal_assists_df

,Team,RK,Name,P,A
0,Arsenal,1.0,Gabriel Magalhães,11,2
1,Arsenal,1.0,Declan Rice,11,2
2,Arsenal,1.0,Riccardo Calafiori,11,2
3,Arsenal,1.0,Leandro Trossard,9,2
4,Arsenal,1.0,Eberechi Eze,10,2
5,Arsenal,6.0,Martín Zubimendi,11,1
6,Arsenal,6.0,Jurriën Timber,11,1
7,Arsenal,6.0,Mikel Merino,10,1
8,Arsenal,6.0,Martin Ødegaard,6,1
9,Arsenal,10.0,David Raya,11,0


In [23]:
Arsenal_discipline_df

,Team,RK,Name,P,yc,rc,Pts
0,Arsenal,1.0,Riccardo Calafiori,11,3,0,3
1,Arsenal,2.0,Martín Zubimendi,11,2,0,2
2,Arsenal,2.0,Jurriën Timber,11,2,0,2
3,Arsenal,4.0,Viktor Gyökeres,10,1,0,1
4,Arsenal,4.0,Myles Lewis-Skelly,7,1,0,1
5,Arsenal,4.0,Gabriel Magalhães,11,1,0,1
6,Arsenal,4.0,David Raya,11,1,0,1
7,Arsenal,8.0,Gabriel Martinelli,8,0,0,0
8,Arsenal,8.0,Kai Havertz,1,0,0,0
9,Arsenal,8.0,Leandro Trossard,9,0,0,0


In [24]:
Arsenal_goals_records_df

,Team,category,Goals,MATCH,Dates
0,Arsenal,Goals Scored (H),5,ARSArsenal5 - 0LEELeeds United,"Sat, Aug 23"
1,Arsenal,Goals Scored (A),2,NEWNewcastle United1 - 2ARSArsenal,"Sun, Sep 28"
2,Arsenal,Goals Conceded (H),1,ARSArsenal1 - 1MNCManchester City,"Sun, Sep 21"
3,Arsenal,Goals Conceded (A),2,SUNSunderland2 - 2ARSArsenal,"Sat, Nov 8"
4,Arsenal,Winning Margin,5,ARSArsenal5 - 0LEELeeds United,"Sat, Aug 23"
5,Arsenal,Losing Margin,1,LIVLiverpool1 - 0ARSArsenal,"Sun, Aug 31"


In [25]:
Arsenal_streaks_df

,Team,category,Games
0,Arsenal,Longest Winning,5
1,Arsenal,Longest Current Winning,0
2,Arsenal,Longest Unbeaten,2
3,Arsenal,Longest Current Unbeaten,8
4,Arsenal,Longest Losing,1
5,Arsenal,Longest Current Losing,0
6,Arsenal,Longest Winless,1
7,Arsenal,Longest Current Winless,1


In [27]:
Arsenal_attendance_df

,Team,category,ATT,MATCH,Dates
0,Arsenal,Largest Attendance,60181,ARSArsenal2 - 0WHUWest Ham United,"Sat, Oct 4"
1,Arsenal,Lowest Attendance,60103,ARSArsenal1 - 0CRYCrystal Palace,"Sun, Oct 26"
2,Arsenal,Average Attendance,60144,-,-
3,Arsenal,Aggregated Attendance,300722,-,-
